# Baseline Bayesian agent on a Watts–Strogatz network

Agents update beliefs in log-odds space:

$$\ell_i \leftarrow \ell_i + \text{gain} \cdot w_{\text{novelty}} \cdot w_{\text{compat}} \cdot w_{\text{source}} \cdot w_{\text{salience}} \cdot \text{LLR}(c)$$

**Baseline:** all weights = 1 except $w_{\text{novelty}}$ = 0 on repeated claims (normative de-duplication). No biases active.

**Per-step loop:** reception → belief update → emission
- *Reception* (`select_neighbor`): agent reads the last broadcast of a random neighbour
- *Update* (`step`): Bayesian log-odds update, claim marked as seen
- *Emission* (`emit`): agent stochastically samples one claim from its repertoire, weighted toward sign-agreement with its current belief $\ell_i$

In [ ]:
from dataclasses import replace
from rankers import Config, run, run_replicates, select_neighbor, select_external
import numpy as np
import matplotlib.pyplot as plt

## 1 — Single run

In [ ]:
cfg = Config(
    n                    = 2_000,
    k                    = 6,
    p_rewire             = 0.1,
    n_claims             = 200,
    llr_std              = 1.0,
    repertoire_seed_size = 5,     # claims each agent knows at t=0
    emission_temp        = 1.0,   # 0 = uniform sharing, higher = more opinion-driven
    n_steps              = 500,   # increase for production; emit() costs ~2 ms/step
    seed                 = 42,
)

result = run(cfg, selection=select_neighbor)
print(f"Elapsed          : {result.elapsed_s:.3f} s")
print(f"Final variance   : {result.history['variance'][-1]:.4f}")
print(f"Final bimodality : {result.history['bimodality'][-1]:.4f}  (threshold 0.555)")

In [ ]:
t = np.arange(len(result.history['variance']))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f"Baseline — WS(N={cfg.n}, k={cfg.k}, p={cfg.p_rewire})  |  single run", y=1.01)

axes[0].plot(t, result.history['variance'], color='steelblue')
axes[0].set(title='Belief variance', xlabel='Step', ylabel='Var($\\ell$)')

axes[1].plot(t, result.history['bimodality'], color='darkorange')
axes[1].axhline(5/9, ls='--', color='gray', lw=1, label='threshold 5/9')
axes[1].set(title='Bimodality coefficient $B$', xlabel='Step')
axes[1].legend(fontsize=8)

axes[2].hist(result.final_beliefs, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[2].axvline(0, ls='--', color='red', alpha=0.7, lw=1)
axes[2].set(title='Final belief distribution', xlabel='Log-odds $\\ell$', ylabel='count')

plt.tight_layout()
plt.show()

## 2 — Repertoire growth per agent

How many distinct arguments does each agent accumulate over time?  
Each line is one of 100 randomly sampled agents; the bold line is the population mean.  
Agents are coloured by their **final belief** (blue → negative, red → positive) so we can see whether opinion strength correlates with information intake.

In [ ]:
r_tracked = run(cfg, track_agents=100)

rep  = r_tracked.repertoire_history   # (n_steps, 100)
ids  = r_tracked.tracked_agents       # (100,)
t    = np.arange(rep.shape[0])

# Colour agents by final belief — diverging: blue=negative, red=positive
final_b  = r_tracked.final_beliefs[ids]
vmax     = np.abs(final_b).max()
norm     = plt.Normalize(-vmax, vmax)
cmap     = plt.cm.RdBu_r

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Repertoire growth — WS(N={cfg.n}, k={cfg.k}, p={cfg.p_rewire})  |  "
    f"pool size M={cfg.n_claims}  seed={cfg.repertoire_seed_size}",
    y=1.01,
)

# ── Left: trajectories ────────────────────────────────────────────────────────
ax = axes[0]
for i in range(rep.shape[1]):
    ax.plot(t, rep[:, i], color=cmap(norm(final_b[i])), alpha=0.25, lw=0.8)
ax.plot(t, rep.mean(axis=1), color="black", lw=2, label="mean")
ax.axhline(cfg.repertoire_seed_size, ls=":", color="gray", lw=1, label="seed size")
ax.axhline(cfg.n_claims, ls="--", color="gray", lw=1, label=f"pool size ({cfg.n_claims})")
ax.set(xlabel="Step", ylabel="Claims in repertoire", title="Repertoire size over time")
ax.legend(fontsize=8)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
fig.colorbar(sm, ax=ax, label="Final belief $\\ell$", shrink=0.8)

# ── Right: final repertoire size vs final belief ──────────────────────────────
ax2 = axes[1]
final_rep = rep[-1]   # (100,)
sc = ax2.scatter(final_b, final_rep, c=final_b, cmap=cmap, norm=norm,
                 edgecolors="white", linewidths=0.4, s=40, alpha=0.8)
ax2.axvline(0, ls="--", color="gray", lw=1, alpha=0.6)
ax2.set(xlabel="Final belief $\\ell$", ylabel="Final repertoire size",
        title="Repertoire size vs belief at $t_{\\mathrm{end}}$")
fig.colorbar(sc, ax=ax2, label="Final belief $\\ell$", shrink=0.8)

plt.tight_layout()
plt.show()

print(f"Mean final repertoire : {final_rep.mean():.1f} / {cfg.n_claims} claims  "
      f"({final_rep.mean()/cfg.n_claims*100:.1f}%)")
print(f"Min / Max             : {final_rep.min()} / {final_rep.max()}")

## 3 — Network vs. external selection

**`select_neighbor`**: each agent receives the last-broadcast claim of a random neighbour — network topology shapes exposure.  
**`select_external`**: each agent draws independently from the full claim pool — no network effect.  

Comparing the two isolates the structural contribution of the WS graph.

In [ ]:
N_REPS = 20

agg_net = run_replicates(cfg, n_reps=N_REPS, selection=select_neighbor)
agg_ext = run_replicates(cfg, n_reps=N_REPS, selection=select_external)

print(f"Network  final variance: {agg_net['mean']['variance'][-1]:.4f}")
print(f"External final variance: {agg_ext['mean']['variance'][-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f"Network vs. external selection  —  {N_REPS} replicates each", y=1.01)

for ax, key, ylabel in zip(
    axes,
    ['variance', 'bimodality'],
    ['Var($\\ell$)', 'Bimodality $B$'],
):
    for agg, label, color in [
        (agg_net, 'network',  'steelblue'),
        (agg_ext, 'external', 'tomato'),
    ]:
        mu = agg['mean'][key]
        sd = agg['std'][key]
        tt = np.arange(len(mu))
        ax.plot(tt, mu, color=color, label=label)
        ax.fill_between(tt, mu - sd, mu + sd, color=color, alpha=0.15)

    if key == 'bimodality':
        ax.axhline(5/9, ls='--', color='gray', lw=1, label='threshold')

    ax.set(title=ylabel, xlabel='Step', ylabel=ylabel)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4 — Final belief distributions across replicates

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
fig.suptitle('Final belief distributions across replicates', y=1.01)

for ax, agg, title, color in [
    (axes[0], agg_net, 'Network selection',  'steelblue'),
    (axes[1], agg_ext, 'External selection', 'tomato'),
]:
    beliefs = agg['final_beliefs'].ravel()
    ax.hist(beliefs, bins=80, color=color, edgecolor='white', linewidth=0.2, density=True)
    ax.axvline(0, ls='--', color='black', alpha=0.5, lw=1)
    ax.set(title=title, xlabel='Log-odds $\\ell$', ylabel='density')

plt.tight_layout()
plt.show()

## 5 — Emission temperature

The `emission_temp` parameter $\beta$ controls how strongly agents prefer to broadcast claims that agree with their current belief:

$$w(c, i) \propto \exp(\beta \cdot \ell_i \cdot \text{LLR}(c)) \quad \text{for } c \in \text{repertoire}(i)$$

- $\beta = 0$: uniform random from repertoire (no opinion influence on sharing)
- $\beta > 0$: preferentially share belief-aligned claims
- $\beta \to \infty$: always broadcast the most belief-aligned claim in repertoire

In [ ]:
temps = [0.0, 0.5, 1.0, 2.0, 4.0]
N_REPS_ET = 10

results_by_temp = {}
for beta in temps:
    agg = run_replicates(replace(cfg, emission_temp=beta, seed=0), n_reps=N_REPS_ET)
    results_by_temp[beta] = agg
    print(f"beta={beta:.1f}  var={agg['mean']['variance'][-1]:.3f}  B={agg['mean']['bimodality'][-1]:.3f}")

In [ ]:
cmap = plt.cm.viridis
colors = [cmap(i / (len(temps) - 1)) for i in range(len(temps))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle(f'Effect of emission temperature $\\beta$  —  {N_REPS_ET} replicates each', y=1.01)

for ax, key, ylabel in zip(axes, ['variance', 'bimodality'], ['Var($\\ell$)', 'Bimodality $B$']):
    for beta, color in zip(temps, colors):
        agg = results_by_temp[beta]
        mu  = agg['mean'][key]
        tt  = np.arange(len(mu))
        ax.plot(tt, mu, color=color, label=f'$\\beta$={beta}')
    if key == 'bimodality':
        ax.axhline(5/9, ls='--', color='gray', lw=1, label='threshold')
    ax.set(title=ylabel, xlabel='Step', ylabel=ylabel)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 6 — Sensitivity: rewiring probability

How does WS topology (from ring lattice at $p=0$ to random graph at $p=1$) affect polarisation?

In [ ]:
p_values   = [0.0, 0.01, 0.05, 0.1, 0.3, 1.0]
N_REPS_SEN = 10

final_var  = []
final_bimo = []

for p in p_values:
    agg = run_replicates(
        replace(cfg, p_rewire=p, seed=0),
        n_reps=N_REPS_SEN,
        selection=select_neighbor,
    )
    final_var.append(agg['mean']['variance'][-1])
    final_bimo.append(agg['mean']['bimodality'][-1])
    print(f"p={p:.2f}  var={final_var[-1]:.3f}  B={final_bimo[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Sensitivity to WS rewiring probability $p$', y=1.01)

axes[0].plot(p_values, final_var, 'o-', color='steelblue')
axes[0].set(title='Final variance', xlabel='Rewiring probability $p$', ylabel='Var($\\ell$)')

axes[1].plot(p_values, final_bimo, 'o-', color='darkorange')
axes[1].axhline(5/9, ls='--', color='gray', lw=1, label='threshold')
axes[1].set(title='Final bimodality $B$', xlabel='Rewiring probability $p$')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()